# Train classifier (EAF GPU)

**Kernel:** `conda env:.conda-diffusion`

Classifier is trained on diffusion-noised images (see `scripts/classifier_train.py` in diffusion-anomaly).


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

APP_ROOT = Path("/exp/sbnd/app/users/munjung/anomaly-detection")
sys.path.insert(0, str(APP_ROOT))
from configs.paths import DATA_ROOT, SCRATCH_TRAINING, ensure_layout

ensure_layout()
discover = {}
for p in [DATA_ROOT / "training" / "eaf_discover.json", APP_ROOT / "train" / "eaf_discover.json"]:
    if p.is_file():
        discover = json.loads(p.read_text())
        break

DIFFUSION_ROOT = Path(discover.get("diffusion_root") or (APP_ROOT / "train" / "diffusion-anomaly"))
FLAG_FILE = DIFFUSION_ROOT / "classifier_flags.sh"
DATA_DIR = Path(discover.get("scratch_anomaly", "/scratch/7DayLifetime/munjung/anomaly-detection")) / "npz"
LOG_DIR = SCRATCH_TRAINING / "classifier" / "run_nominal"
LOG_DIR.mkdir(parents=True, exist_ok=True)
print("DIFFUSION_ROOT", DIFFUSION_ROOT)
print("FLAG_FILE", FLAG_FILE, "exists", FLAG_FILE.exists())
print("DATA_DIR", DATA_DIR)
print(FLAG_FILE.read_text() if FLAG_FILE.exists() else "(missing flags)")


In [ ]:
cmd = f'''
set -e
cd "{DIFFUSION_ROOT}"
source "{FLAG_FILE}"
export CLASSIFIER_TRAIN_FLAGS="$CLASSIFIER_TRAIN_FLAGS --data_dir {DATA_DIR}"
echo "CLASSIFIER_TRAIN_FLAGS=$CLASSIFIER_TRAIN_FLAGS"
# python scripts/classifier_train.py $CLASSIFIER_TRAIN_FLAGS 2>&1 | tee "{LOG_DIR}/train.log"
echo "(dry-run) uncomment to train"
'''
print(cmd)
